# Seas of Data: train the ship detector (HRSID -> YOLO11)
Runtime > Change runtime type > **T4 GPU**, then Run all. Takes ~55 min.

**Weights are saved to your Google Drive** (`MyDrive/seas_of_data`) as training runs, so a Colab reset cannot lose them. Click *Allow* on the Drive pop-up in the first cell. Keep this tab open.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/seas_of_data
!nvidia-smi | head -15
!pip -q install ultralytics gdown

In [ ]:
!gdown 1NY3ovgc-woDlNoQdyqzRB3t9McOBH5Ms -O /content/hrsid_jpg.zip
!unzip -q -o /content/hrsid_jpg.zip -d /content/hrsid_unzipped
!ls /content/hrsid_unzipped

In [ ]:
%%writefile /content/prepare_hrsid.py
"""Convert HRSID (COCO json + images) into a YOLO dataset under ml/data/hrsid/.

Usage: .venv/bin/python prepare_hrsid.py <unzipped_hrsid_dir>
Official split: train + test json. We carve 10% of train into val (seeded).
"""
import json
import random
import shutil
import sys
from pathlib import Path

root = Path(sys.argv[1])
out = Path(__file__).parent / "data" / "hrsid"
random.seed(0)

images = {p.name: p for p in root.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}}
jsons = {p.name.lower(): p for p in root.rglob("*.json")}
print(f"found {len(images)} images, jsons: {sorted(jsons)}")


def pick(name):
    if name not in jsons:
        raise SystemExit(f"missing {name}; have {sorted(jsons)}")
    return jsons[name]


def load(path):
    d = json.loads(path.read_text())
    by_img = {}
    for a in d["annotations"]:
        by_img.setdefault(a["image_id"], []).append(a)
    return [(im, by_img.get(im["id"], [])) for im in d["images"]]


def write(split, items):
    (out / "images" / split).mkdir(parents=True, exist_ok=True)
    (out / "labels" / split).mkdir(parents=True, exist_ok=True)
    n_box = skipped = 0
    for im, anns in items:
        src = images.get(Path(im["file_name"]).name)
        if src is None:
            skipped += 1
            continue
        w, h = im["width"], im["height"]
        lines = []
        for a in anns:
            x, y, bw, bh = a["bbox"]
            if bw <= 0 or bh <= 0:
                continue
            lines.append(f"0 {(x + bw / 2) / w:.6f} {(y + bh / 2) / h:.6f} {bw / w:.6f} {bh / h:.6f}")
        shutil.copy(src, out / "images" / split / src.name)
        (out / "labels" / split / (src.stem + ".txt")).write_text("\n".join(lines))
        n_box += len(lines)
    print(f"{split}: {len(items) - skipped} images, {n_box} boxes, {skipped} missing images")


train = load(pick("train2017.json"))
test = load(pick("test2017.json"))
random.shuffle(train)
k = len(train) // 10
write("val", train[:k])
write("train", train[k:])
write("test", test)

(out / "hrsid.yaml").write_text(
    f"path: {out.resolve()}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: ship\n"
)
print("wrote", out / "hrsid.yaml")

In [ ]:
!mkdir -p /content/ml && cp /content/prepare_hrsid.py /content/ml/ && cd /content/ml && python prepare_hrsid.py /content/hrsid_unzipped
!cat /content/ml/data/hrsid/hrsid.yaml

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11s.pt')
# project is on Drive: best.pt and last.pt are written there every epoch
model.train(data='/content/ml/data/hrsid/hrsid.yaml', epochs=50, imgsz=640, batch=32, patience=10, project='/content/drive/MyDrive/seas_of_data', name='hrsid_s', exist_ok=True, plots=False)

In [ ]:
# honest numbers on the held-out official TEST split (never seen in training)
import json
W = '/content/drive/MyDrive/seas_of_data/hrsid_s/weights/best.pt'
best = YOLO(W)
m = best.val(data='/content/ml/data/hrsid/hrsid.yaml', split='test', imgsz=640, plots=False)
res = {'model':'yolo11s','dataset':'HRSID official test split','n_images':1962,'precision':float(m.box.mp),'recall':float(m.box.mr),'mAP50':float(m.box.map50),'mAP50_95':float(m.box.map)}
json.dump(res, open('/content/drive/MyDrive/seas_of_data/metrics.json','w'), indent=2); print(res)

In [ ]:
# copies are already on Drive; also download straight to the computer
import shutil
from google.colab import files
shutil.copy('/content/drive/MyDrive/seas_of_data/hrsid_s/weights/best.pt', '/content/best.pt')
files.download('/content/best.pt')
files.download('/content/drive/MyDrive/seas_of_data/metrics.json')

After the downloads: put `best.pt` in `ml/models/` and `metrics.json` in `ml/reports/` in the repo.

**If Colab disconnects mid-training:** reconnect, rerun the first four cells (Drive, download, prepare), then run this and continue with the cells below it:
```
from ultralytics import YOLO
YOLO('/content/drive/MyDrive/seas_of_data/hrsid_s/weights/last.pt').train(resume=True)
```